- 

# 1.1 torch的语音数据集

In [1]:
from torchaudio.datasets import SPEECHCOMMANDS

da_train = SPEECHCOMMANDS(root="/Users/logicye/Code/Datasets/", download=False, subset="training")         #
da_valid = SPEECHCOMMANDS(root="/Users/logicye/Code/Datasets/", download=False, subset="validation")       #
da_test = SPEECHCOMMANDS(root="/Users/logicye/Code/Datasets/", download=False, subset="testing")            #
print(f"训练集样本数量: {len(da_train)}")
print(f"验证集样本数量: {len(da_valid)}")
print(f"测试集样本数量: {len(da_test)}")


    # traing: 用于训练模型的数据集
    # validation: 用于验证模型性能的数据集
    # testing: 用于评估模型最终性能的数据集（损失值、准确率等指标）

    # ds_train[0]      #读取训练集中的第一个样本，返回一个元组，包含音频数据、采样率、标签、说话者ID和样本路径等信息。
    # ds_train[0][0]   #读取训练集中的第一个样本的音频数据部分，返回一个张量，表示音频信号的数值。
    # ds_train[0][0].shape  #第一维表示音频信号的通道数（通常为1，表示单声道；2表示双通道），第二维表示音频信号的长度（以采样点为单位）。
    # 注意把多通道音频数据转换为单通道，即使用单声道音频数据进行训练，通常可以通过取平均值或选择一个通道来实现。




训练集样本数量: 84843
验证集样本数量: 9981
测试集样本数量: 11005


In [5]:
import torch
torch.Size([1, 16000])  #表示音频数据是单声道的，采样率为16000Hz，每个样本包含16000个采样点，代表1秒钟的音频数据。


torch.Size([1, 16000])

# 可视化
import matplotlib.pyplot as plt

plt.plt(ds_train[0][0][0],color="red")

In [10]:
import torchaudio
torchaudio.save("output.wav", da_train[0][0], 16000)  #将训练集中的第一个样本的音频数据保存为一个名为"output.wav"的音频文件，采样率为16000Hz。

/Users/logicye/Code/ai_learning/learing_venv/lib/python3.13/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/Users/logicye/Code/ai_learning/learing_venv/lib/python3.13/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch

# 1.2 语音文件

- 可以使用处理语音的模块
    - pip install soundfile

- 两个：
    - soundfile
    - torchaudio

In [2]:
!pip install soundfile  #安装soundfile库，用于处理音频文件的读写操作。

  Using cached soundfile-0.13.1-py2.py3-none-macosx_11_0_arm64.whl.metadata (16 kB)
  Using cached cffi-2.0.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached soundfile-0.13.1-py2.py3-none-macosx_11_0_arm64.whl (1.1 MB)
Using cached cffi-2.0.0-cp313-cp313-macosx_11_0_arm64.whl (181 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [soundfile]


In [ ]:
# 演示两种用法
from matplotlib import pyplot as plt
import soundfile
import torchaudio

# 🔥 加这一行！Mac 必加，解决后端问题
torchaudio.set_audio_backend("soundfile")

# 读文件
wav1, sample_rate = soundfile.read("1.wav")
print(wav1, sample_rate)
plt.plot(wav1, color="red")

wav2, s_rate = torchaudio.load("1.wav")
print(wav2, s_rate)

soundfile.write("2.wav", wav2[0].numpy(), 16000)  # 我也帮你修正了这里

# 2.处理器与模型
## 2.1使用处理器的方式

In [8]:
from transformers import ASTFeatureExtractor
# ASTFeatureExtractor是一个用于音频特征提取的类，提供了多种方法来处理和转换音频数据，以便于后续的模型训练和推理。

# 尝试定制构造一个
feature_extractor = ASTFeatureExtractor(
    feature_size=1,         #特征维度，表示每个时间步的特征向量的维度。
    sampling_rate=16000,    #采样率，表示音频数据的采样频率，单位为赫兹（Hz）。
    hop_length=160,         #跳跃长度，表示在提取特征时，每次移动的采样点数。
    #num_mel_bins=128,      #梅尔频率滤波器组的数量，表示将频谱转换为梅尔频率域时使用的滤波器数量。
    n_mels=128,             #梅尔频率滤波器组的数量，表示将频谱转换为梅尔频率域时使用的滤波器数量。

    max_length=128,         #最大长度，表示在提取特征时考虑的最大音频长度，单位为采样点数。
    do_normalize=True,      #是否对提取的特征进行归一化处理，通常是将特征值缩放到一个特定的范围内，以提高模型的训练效果。
    mean=-6.845978,        #特征归一化的均值，表示在进行归一化处理时使用的均值值。
    std=5.5654526,           #特征归一化的标准差，
    #fmin=0,                 #最低频率，表示在提取特征时考虑的最低频率，单位为赫兹（Hz）。
    #fmax=8000,              #最高频率，表示在提取特征时考虑的最高频率，单位为赫兹（Hz）。

    return_attention_mask=False,  #是否返回注意力掩码，通常用于处理变长输入的情况，以指示哪些部分是有效的输入数据。
)

#上面就等价于 ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")  

"""
    测试效果：

"""

/Users/logicye/Code/ai_learning/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'\n    测试效果：\n\n'

In [9]:
import soundfile
# 读取音频文件
wav , sample_rate = soundfile.read("1.wav")
#print(wav, sample_rate)

# 提取特征
input_features = feature_extractor(wav, return_tensors="pt")
print(input_features)


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


{'input_values': tensor([[[-0.0850, -0.4095, -0.1002,  ..., -0.0236, -0.0825, -0.1853],
         [-0.0716, -0.3139, -0.0045,  ..., -0.0698, -0.0979, -0.1038],
         [-0.0584, -0.3728, -0.0635,  ..., -0.0240, -0.0142, -0.1508],
         ...,
         [ 0.6150,  0.6150,  0.6150,  ...,  0.6150,  0.6150,  0.6150],
         [ 0.6150,  0.6150,  0.6150,  ...,  0.6150,  0.6150,  0.6150],
         [ 0.6150,  0.6150,  0.6150,  ...,  0.6150,  0.6150,  0.6150]]])}


# 

In [10]:
from transformers import ASTForAudioClassification
import torch

# 加载模型
model = ASTForAudioClassification.from_pretrained("MIT/ast-finetuned-speech-commands-v2", 
                                                  #dtype=torch.float32,            #指定模型权重的数据类型为torch.float32，以确保在推理过程中使用单精度浮点数进行计算，这有助于提高模型的性能和效率。  
                                                  #dtype=torch.float16             #指定模型权重的数据类型为torch.float16，预测精度下降，但可以接受
                                                  )

# 使用上面提取的特征进行推理
#outputs = model(input_values=input_features["input_values"])
outputs = model(**input_features)                                                   #解包输入特征字典，将其作为位置参数传递给模型的forward方法。
print(outputs)
print(outputs.logits)       #模型的输出包含一个logits属性，表示模型对每个类别的预测分数。通过访问outputs.logits，可以获取这些预测分数，以便进行后续的处理和分析。    

# 处理结果

SequenceClassifierOutput(loss=None, logits=tensor([[  5.3478,  -9.8538,  -8.5550,  -9.0400,  -9.5786,  -9.5634,  -9.9788,
          -9.4335,  -9.9956,  -9.5568, -10.0332,  -9.6417, -10.1127,  -9.2160,
          -9.3190,  -9.3526, -10.5328,  -8.0418,  -9.3034, -11.3892,  -9.4785,
          -9.0596,  -9.1164,  -8.6740,  -9.3100,  -9.7137,  -7.9949,  -9.2152,
          -9.8133,  -9.2026, -10.3113,  -8.5308, -10.2963,  -9.6160,  -9.7845]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
tensor([[  5.3478,  -9.8538,  -8.5550,  -9.0400,  -9.5786,  -9.5634,  -9.9788,
          -9.4335,  -9.9956,  -9.5568, -10.0332,  -9.6417, -10.1127,  -9.2160,
          -9.3190,  -9.3526, -10.5328,  -8.0418,  -9.3034, -11.3892,  -9.4785,
          -9.0596,  -9.1164,  -8.6740,  -9.3100,  -9.7137,  -7.9949,  -9.2152,
          -9.8133,  -9.2026, -10.3113,  -8.5308, -10.2963,  -9.6160,  -9.7845]],
       grad_fn=<AddmmBackward0>)


## 2.2 使用torchaudio进行推理

- 在transformers 找到现成的语音命令模型
- 找到类似的任务，确定算法与模型
- 采集数据集，构建数据集工程
- 训练
- 封装推理XXXXPipeline
- 应用

In [11]:
import torchaudio

wav_torch, sample_rate_torch = torchaudio.load("1.wav")
#print(wav_torch,sample_rate_torch)

#input_features_torch = feature_extractor(wav_torch[0], return_tensors="pt")
input_features_torch = feature_extractor(wav_torch.squeeze(0),return_tensors="pt")  
#print(input_features_torch)

###################################################
# 判定设备
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)  #将模型移动到GPU上进行计算，以加速推理过程。

###################################################

logits = model(**input_features_torch.to(device)).logits.argmax(1).cpu().item()  #使用torch.argmax函数来获取模型输出的logits张量中每个样本的最大值所在的索引位置，dim=-1表示在最后一个维度上进行操作。
model.config.id2label[logits]                                         #根据模型的配置文件中的id2label字典，将预测的类别索引转换为对应的标签名称，以便更直观地理解模型的预测结果。

RuntimeError: Couldn't find appropriate backend to handle uri 1.wav and format None.